In [26]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 
from uuid import uuid4
import json

### Build tool registry JSON from function definitions.
The LLM can search the registry, fetch and execute scripts, and push new scripts to the registry


In [27]:
from src.tool_registry import ScriptRegistry, build_registry_tools

registry = ScriptRegistry()
tools = build_registry_tools(registry)

# After reviewing a generated script, enable it explicitly:
# registry.enable_script("the-id-returned-by-write_script_tool")


In [28]:
chat = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    max_tokens=2048,
)

SystemPrompt = """You are a voice assistant with a persistent Python tool registry.
Search with search_scripts before creating a tool. Use execute_script with an
existing enabled tool's ID and schema-compatible arguments when it fits.
If no suitable tool exists, use write_script_tool to save a new script. Give
its entry-point function typed named parameters, a return annotation, a clear
docstring, and a JSON-serializable result. script_name has no path or extension.
New scripts are pending_validation: explain that review and enabling are needed
before execution. Never claim a pending script has run. Tool failures are
returned as error objects; read them and correct the request when appropriate.
"""
tools = build_registry_tools(registry)

messages = [HumanMessage(content="Print the word Hell after 3 seconds.")]
messages_write_python_script = [HumanMessage(
    content="Create and register a reusable function that takes two numbers and returns their sum."
)]


def llm_chat(messages: list, tools: list | None = None, max_rounds: int = 8) -> list:
    available = {registered_tool.name: registered_tool for registered_tool in (tools or [])}
    model_with_tools = chat.bind_tools(list(available.values())) if available else chat
    for _ in range(max_rounds):
        result = model_with_tools.invoke([SystemMessage(content=SystemPrompt)] + messages)
        messages.append(result)
        if result.content:
            print(f"[AGENT]: {result.content}")
        if not result.tool_calls:
            return messages
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call['name']}")
            try:
                registered_tool = available[call["name"]]
                tool_result = registered_tool.invoke(call["args"])
            except Exception as exc:
                tool_result = {"error": str(exc)}
            print(f"[OBS]: {tool_result}")
            messages.append(ToolMessage(
                content=json.dumps(tool_result), tool_call_id=call["id"]
            ))
    raise RuntimeError("Agent reached its tool-call round limit.")


In [29]:
result = llm_chat(messages_write_python_script, tools)

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[TOOL CALL]: write_script_tool
[OBS]: {'error': "A tool named 'add_numbers' is already registered."}


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[TOOL CALL]: search_scripts
[OBS]: [{'name': 'add_numbers', 'description': 'Add two numbers and return their sum.', 'parameters': {'additionalProperties': False, 'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}}, 'required': ['a', 'b'], 'title': 'ScriptArguments', 'type': 'object'}, 'return_type': 'int | float', 'id': '6d376059-2bfd-46bc-a2d6-13072db5e683', 'path': 'agent_generated_scripts/add_numbers.py', 'entry_point': 'add_numbers', 'status': 'enabled', 'version': 1, 'sha256': '78040f4232b790714266939210d8edad8c5dcbc6bfbea4a12bb7e28d1b97a781'}]
[AGENT]: A reusable function named `add_numbers` already exists in your registry and is currently **enabled**. Here are its details:

| Property | Value |
|----------|-------|
| **ID** | `6d376059-2bfd-46bc-a2d6-13072db5e683` |
| **Name** | `add_numbers` |
| **Status** | Enabled |
| **Parameters** | `a` (int or float), `b` (int or fl

#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [30]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



In [31]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recording audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sr),
        samplerate=sr,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [32]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 2 μs, sys: 3 μs, total: 5 μs
Wall time: 7.39 μs
[ASRTranscription(language='English', text="Hi, I'm Shane. Welcome to Reddit Stories. Today is a very ancient episode of the show. In honor of Dino Week, we're diving into the.", time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output

In [33]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = llm_chat(messages=messages, tools=tools)


Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: Hello! I'm ready to help you with your tasks. 

I have access to:
- **Search scripts** - Find existing saved scripts by name or description
- **Write scripts** - Create new Python functions with typed parameters
- **Execute scripts** - Run enabled registered functions with JSON arguments

What would you like me to help you with? Please let me know your specific task or question.


In [34]:
llm_result

[HumanMessage(content='', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello! I'm ready to help you with your tasks. \n\nI have access to:\n- **Search scripts** - Find existing saved scripts by name or description\n- **Write scripts** - Create new Python functions with typed parameters\n- **Execute scripts** - Run enabled registered functions with JSON arguments\n\nWhat would you like me to help you with? Please let me know your specific task or question.", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-20T16:45:04.265497632Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2237744561, 'load_duration': 2288468, 'prompt_eval_count': 666, 'prompt_eval_duration': 216681000, 'eval_count': 158, 'eval_duration': 1888367000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a15a33-bac9-70f0-984d-5e023fe2fcaa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 666, '

### The Pipeline looks fairly good, we will now deal with voice verification 

In [35]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('audio_wavs/recording.wav')
print(signal.shape)
signal = signal.mean(dim=0, keepdim=True)
signal = torchaudio.functional.resample(signal, fs, 16000) # necessary for ecapa-tdnn
embeddings = classifier.encode_batch(signal)


Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([1, 120000])


In [53]:
classifier

EncoderClassifier(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentum

In [ ]:
from speechbrain.inference.speaker import SpeakerRecognition
verification = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_dif.wav") # Different Speakers
print(f"{score= }, {prediction= }")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_sim.wav") # Same Speaker
print(f"{score= }, {prediction= }")

Fetch hyperparams.yaml: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/hyperparams.yaml'
Fetch embedding_model.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/embedding_model.ckpt'
Fetch mean_var_norm_emb.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/mean_var_norm_emb.ckpt'
Fetch classifier.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/classifier.ckpt'
Fetch label_encoder.txt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/label_encoder.ckpt'
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


score= tensor([0.2358], device='cuda:0'), prediction= tensor([False], device='cuda:0')
score= tensor([0.7795], device='cuda:0'), prediction= tensor([True], device='cuda:0')


In [52]:
verification

SpeakerRecognition(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentu

In [37]:

import torch
import torch.nn.functional as F 


def enroll_speaker(trials=2, trial_duration=5):
    pass_threshold = 0.45
    print(f"You will be asked to speak {trials} times, each time you can say whatever you want for {trial_duration} seconds, the model will save your voiceprint afterwards")
    embeddings = []
    for i in range(trials):
        input(f"Press Enter to Start trial {i+1}...")
        print(f"Trial number {i+1} : ")
        audio, sr = record_audio(16000, trial_duration)
        audio = torch.tensor(audio, dtype=torch.float32)
        audio = audio.T
        audio = audio.mean(dim=0, keepdim=True) # in case of multiple channels, ecapa accepts mono audio with sr = 16000
        audio = torchaudio.functional.resample(audio, sr, 16000)
        embeddings.append(classifier.encode_batch(audio, normalize=False))
        if i > 0:
            similarity = F.cosine_similarity(embeddings[-1][0,:,:], embeddings[-2][0,:,:])
            assert similarity.item() > pass_threshold, "Your recorded voice is very different from the previous, try again"
    res = torch.cat(embeddings, dim=0)
    res = F.normalize(res, p=2, dim=-1)
    print(f"Finished Enrollment Procedure")
    return F.normalize(res.mean(dim=0), dim=-1)



def authorized_speaker(audio, sr, ref_dim:torch.tensor) -> bool:
    """Checks the similarity with the enrolled speaker """
    if not isinstance(audio, torch.Tensor):
        audio = torch.tensor(audio, dtype=torch.float32)
        if audio.shape[0] != 1:
            audio = audio.T
    print(audio.shape)
    audio = audio.mean(dim=0, keepdim=True)
    audio = torchaudio.functional.resample(audio, sr, 16000)
    audio_dim = classifier.encode_batch(audio, normalize=False)[0]
    audio_dim = F.normalize(audio_dim, p=2, dim=-1)
    sim = F.cosine_similarity(audio_dim, ref_dim)
    print(f"{sim=}")
    return sim.item() > 0.45




In [38]:
authorized_spk_dim = enroll_speaker(trials=3, trial_duration=5)

You will be asked to speak 3 times, each time you can say whatever you want for 5 seconds, the model will save your voiceprint afterwards


Trial number 1 : 
Recording audio for 5 seconds...
Trial number 2 : 
Recording audio for 5 seconds...
Trial number 3 : 
Recording audio for 5 seconds...
Finished Enrollment Procedure


In [46]:
audio = record_audio(16000, 5)

Recording audio for 5 seconds...


In [47]:
authorized_speaker(audio[0], audio[1], authorized_spk_dim)

torch.Size([1, 80000])
sim=tensor([0.0292], device='cuda:0')


False

### The speaker verification is working fine for now (needs further testing to check the authorization).
### The next step is to add it to the pipeline : Audio -> Speaker authorization -> ASR -> LLM -> Response

In [25]:
audio = record_audio(16000, 10)
if not authorized_speaker(audio[0], audio[1], authorized_spk_dim):
    raise Exception('Unauthorized speaker.')
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content=text)]
tools = build_registry_tools(registry)
response = llm_chat(messages=messages, tools=tools)

Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


torch.Size([1, 160000])
sim=tensor([0.7155], device='cuda:0')


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: Hello Shane! Welcome to Reddit Stories. It's great to have you here today. 

Dino Week is a fantastic theme - dinosaurs are such an exciting topic for storytelling! I'm ready to help you create content or scripts for your show. 

What would you like to work on today? Are you looking to:
- Create a new script about dinosaurs?
- Search for existing scripts related to dinosaurs?
- Or perhaps something else entirely?

Let me know how I can assist with your Reddit Stories episode!


In [54]:
import speech_recongition as sr

ModuleNotFoundError: No module named 'speech_recongition'